In [5]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta

np.random.seed(42)

print("Завантаження даних НБУ...")

dates = pd.date_range(start="2025-01-01", end="2025-12-31", freq="MS")
nbu_data = []

for dt in dates:
    date_str = dt.strftime("%Y%m%d")
    url = f"https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?valcode=USD&date={date_str}&json"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200 and len(response.json()) > 0:
            rate = response.json()[0]['rate']
        else:
            rate = 41.5 + np.random.normal(0, 0.5) 
    except Exception:
        rate = 41.5 + np.random.normal(0, 0.5)
    
    nbu_data.append({"month": dt.strftime("%Y-%m"), "usd_rate": round(rate, 2)})

df_macro = pd.DataFrame(nbu_data)
df_macro['key_rate'] = 15.5
df_macro['inflation_yoy'] = [12.1, 11.8, 11.2, 10.5, 9.8, 9.5, 9.1, 8.8, 8.5, 8.2, 8.0, 7.8]

print("Макродані НБУ успішно завантажено:")
print(df_macro.head(3))

print("\nГенерація портфеля боржників...")

n_loans = 1000
loan_ids = [f"NPL-{10000 + i}" for i in range(n_loans)]

segments = ['Consumer_Micro', 'Cash_Loan', 'Auto_Legacy']
segment_weights = [0.5, 0.35, 0.15]

regions = ['Київська', 'Дніпропетровська', 'Львівська', 'Одеська', 'Харківська']

portfolio = []
start_date = datetime(2024, 1, 1)

for l_id in loan_ids:
    seg = np.random.choice(segments, p=segment_weights)
    reg = np.random.choice(regions)
    
    if seg == 'Consumer_Micro':
        principal = np.random.randint(3000, 25000)
    elif seg == 'Cash_Loan':
        principal = np.random.randint(20000, 100000)
    else:
        principal = np.random.randint(80000, 300000)

    random_days = np.random.randint(0, 365)
    dpd_date = start_date + timedelta(days=random_days)
    
    portfolio.append({
        'loan_id': l_id,
        'segment': seg,
        'region': reg,
        'principal_amount': principal,
        'npl_date': dpd_date.strftime("%Y-%m-%d"),
        'cohort_month': dpd_date.strftime("%Y-%m")
    })

df_portfolio = pd.DataFrame(portfolio)


print("Генерація платежів (Прогноз vs Факт)...")

payments = []
observation_months = df_macro['month'].tolist()

for _, loan in df_portfolio.iterrows():
    l_id = loan['loan_id']
    principal = loan['principal_amount']
    seg = loan['segment']
    cohort = loan['cohort_month']

    base_recovery_rate = 0.035 if seg == 'Consumer_Micro' else (0.025 if seg == 'Cash_Loan' else 0.015)
    
    for m in observation_months:
        if m >= cohort:  
            months_in_npl = (int(m[:4]) - int(cohort[:4])) * 12 + (int(m[5:]) - int(cohort[5:])) + 1
            decay_factor = 0.85 ** (months_in_npl - 1)  
            
            forecast_pmt = round(principal * base_recovery_rate * decay_factor, 2)
          
            seasonality = 0.70 if m.endswith('-01') else 1.0

            if seg == 'Consumer_Micro' and m in ['2025-06', '2025-07', '2025-08']:
                seg_factor = 0.75
            else:
                seg_factor = 1.0
                

            random_noise = np.random.uniform(0.85, 1.15)
            
            actual_pmt = round(forecast_pmt * seasonality * seg_factor * random_noise, 2)
            
            payments.append({
                'loan_id': l_id,
                'payment_month': m,
                'forecast_payment': forecast_pmt,
                'actual_payment': actual_pmt
            })

df_payments = pd.DataFrame(payments)

df_portfolio.to_csv('portfolio.csv', index=False)
df_payments.to_csv('payments.csv', index=False)
df_macro.to_csv('nbu_macro.csv', index=False)

print("\nДані успішно згенеровані та збережені у CSV-файли:")
print(f"- portfolio.csv: {len(df_portfolio)} записів")
print(f"- payments.csv: {len(df_payments)} записів")
print(f"- nbu_macro.csv: {len(df_macro)} записів")

Завантаження даних НБУ...
Макродані НБУ успішно завантажено:
     month  usd_rate  key_rate  inflation_yoy
0  2025-01     42.03      15.5           12.1
1  2025-02     41.82      15.5           11.8
2  2025-03     41.51      15.5           11.2

Генерація портфеля боржників...
Генерація платежів (Прогноз vs Факт)...

Дані успішно згенеровані та збережені у CSV-файли:
- portfolio.csv: 1000 записів
- payments.csv: 12000 записів
- nbu_macro.csv: 12 записів


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('npl_analytics.db')
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")
cursor.execute("DROP TABLE IF EXISTS payments;")
cursor.execute("DROP TABLE IF EXISTS portfolio;")
cursor.execute("DROP TABLE IF EXISTS macro;")

cursor.execute('''
CREATE TABLE macro (
    month TEXT PRIMARY KEY,
    usd_rate REAL NOT NULL,
    key_rate REAL NOT NULL,
    inflation_yoy REAL NOT NULL
);
''')

cursor.execute('''
CREATE TABLE portfolio (
    loan_id TEXT PRIMARY KEY,
    segment TEXT NOT NULL,
    region TEXT NOT NULL,
    principal_amount REAL NOT NULL,
    npl_date TEXT NOT NULL,
    cohort_month TEXT NOT NULL
);
''')

cursor.execute('''
CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    loan_id TEXT NOT NULL,
    payment_month TEXT NOT NULL,
    forecast_payment REAL NOT NULL,
    actual_payment REAL NOT NULL,
    FOREIGN KEY (loan_id) REFERENCES portfolio(loan_id) ON DELETE CASCADE
);
''')

conn.commit()

df_portfolio = pd.read_csv('portfolio.csv')
df_payments = pd.read_csv('payments.csv')
df_macro = pd.read_csv('nbu_macro.csv')

df_macro.to_sql('macro', conn, if_exists='append', index=False)
df_portfolio.to_sql('portfolio', conn, if_exists='append', index=False)
df_payments.to_sql('payments', conn, if_exists='append', index=False)

print("Перевірка цілісності бази даних:")
for table in ['macro', 'portfolio', 'payments']:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table};").fetchone()[0]
    print(f" - Таблиця '{table}': {count} рядків")

conn.close()

Перевірка цілісності бази даних:
 - Таблиця 'macro': 12 рядків
 - Таблиця 'portfolio': 1000 рядків
 - Таблиця 'payments': 12000 рядків


In [ ]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('npl_analytics.db')

print("--- 1. ПЕРЕВІРКА НА ПРОПУЩЕНІ ЗНАЧЕННЯ (NULL) ---")
tables = ['portfolio', 'payments', 'macro']

for table in tables:
    df_table = pd.read_sql(f"SELECT * FROM {table}", conn)
    null_counts = df_table.isnull().sum()
    total_nulls = null_counts.sum()
    if total_nulls == 0:
        print(f"Таблиця '{table}': 0 пропусків.")
    else:
        print(f"Таблиця '{table}': виявлено пропуски:")
        print(null_counts[null_counts > 0])


print("\n--- 2. ПЕРЕВІРКА НА ДУБЛІКАТИ КЛЮЧІВ ---")
dup_portfolio = pd.read_sql("SELECT loan_id, COUNT(*) as cnt FROM portfolio GROUP BY loan_id HAVING cnt > 1", conn)
dup_macro = pd.read_sql("SELECT month, COUNT(*) as cnt FROM macro GROUP BY month HAVING cnt > 1", conn)

print(f"Дублікати в portfolio (loan_id): {len(dup_portfolio)}")
print(f"Дублікати в macro (month): {len(dup_macro)}")

print("\n--- 3. ВИЯВЛЕННЯ АНОМАЛІЙ ТА ВИНЯТКІВ (Z-SCORE) ---")
df_payments = pd.read_sql("SELECT * FROM payments", conn)

mean_actual = df_payments['actual_payment'].mean()
std_actual = df_payments['actual_payment'].std()

df_payments['z_score'] = (df_payments['actual_payment'] - mean_actual) / std_actual
outliers = df_payments[np.abs(df_payments['z_score']) > 3]

print(f"Загальна кількість транзакцій: {len(df_payments)}")
print(f"Середній платіж (Mean): {mean_actual:.2f} грн")
print(f"Стандартне відхилення (Std): {std_actual:.2f} грн")
print(f"Виявлено аномальних платежів (|Z-score| > 3): {len(outliers)}")

if len(outliers) > 0:
    print("\nПриклад знайдених аномальних платежів:")
    print(outliers[['payment_id', 'loan_id', 'payment_month', 'actual_payment', 'z_score']].head())

conn.close()

--- 1. ПЕРЕВІРКА НА ПРОПУЩЕНІ ЗНАЧЕННЯ (NULL) ---
Таблиця 'portfolio': 0 пропусків.
Таблиця 'payments': 0 пропусків.
Таблиця 'macro': 0 пропусків.

--- 2. ПЕРЕВІРКА НА ДУБЛІКАТИ КЛЮЧІВ ---
Дублікати в portfolio (loan_id): 0
Дублікати в macro (month): 0

--- 3. ВИЯВЛЕННЯ АНОМАЛІЙ ТА ВИНЯТКІВ (Z-SCORE) ---
Загальна кількість транзакцій: 12000
Середній платіж (Mean): 214.02 грн
Стандартне відхилення (Std): 279.11 грн
Виявлено аномальних платежів (|Z-score| > 3): 278

Приклад знайдених аномальних платежів:
     payment_id    loan_id payment_month  actual_payment   z_score
132         133  NPL-10011       2025-01         1294.26  3.870365
133         134  NPL-10011       2025-02         1773.17  5.586239
134         135  NPL-10011       2025-03         1573.15  4.869592
135         136  NPL-10011       2025-04         1323.65  3.975665
136         137  NPL-10011       2025-05         1119.76  3.245153
